In [44]:
import pandas as pd
#import numpy as np

In [45]:
Data=pd.read_csv("https://17lands-public.s3.amazonaws.com/analysis_data/draft_data/draft_data_public.FDN.TradDraft.csv.gz")

In [46]:
Data.head(15)

,expansion,event_type,draft_id,draft_time,rank,event_match_wins,event_match_losses,pack_number,pick_number,pick,...,pool_Wary Thespian,pool_Wildwood Scourge,pool_Wind-Scarred Crag,pool_Witness Protection,pool_Youthful Valkyrie,"pool_Zimone, Paradox Sculptor",pool_Zombify,"pool_Zul Ashur, Lich Lord",user_n_games_bucket,user_game_win_rate_bucket
0,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,0,Twinflame Tyrant,...,0,0,0,0,0,0,0,0,10,0.34
1,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,1,Solemn Simulacrum,...,0,0,0,0,0,0,0,0,10,0.34
2,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,2,Soulstone Sanctuary,...,0,0,0,0,0,0,0,0,10,0.34
3,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,3,Helpful Hunter,...,0,0,0,0,0,0,0,0,10,0.34
4,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,4,Inspiring Paladin,...,0,0,0,0,0,0,0,0,10,0.34
5,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,5,Macabre Waltz,...,0,0,0,0,0,0,0,0,10,0.34
6,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,6,Luminous Rebuke,...,0,0,0,0,0,0,0,0,10,0.34
7,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,7,Felidar Savior,...,0,0,0,0,0,0,0,0,10,0.34
8,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,8,Blossoming Sands,...,0,0,0,0,0,0,0,0,10,0.34
9,FDN,TradDraft,a3b0379fee704658b8828a02f6b523b5,2024-11-12 20:43:47,NaN,0,3,0,9,Fake Your Own Death,...,0,0,0,0,0,0,0,0,10,0.34


In [47]:
#This is ATA (average taken at), which is very straightforward
Data['pick_number']=Data['pick_number']+1
ATA=Data.groupby('pick').agg(ATA=('pick_number', 'mean')).reset_index()
print(ATA)

                         pick        ATA
0                      Abrade   3.758721
1           Abyssal Harvester   2.241921
2            Adventuring Gear  11.333333
3                Aegis Turtle  11.736021
4                   Aetherize   8.187858
..                        ...        ...
281        Witness Protection   8.889573
282         Youthful Valkyrie   4.633019
283  Zimone, Paradox Sculptor   1.788217
284                   Zombify   7.588393
285      Zul Ashur, Lich Lord   4.728614

[286 rows x 2 columns]


In [48]:
#you could calculate alsa by summing every row's pool_[card]*pick_number
#this is complicated by when something wheels, you only count the second time 
#can calculate summing each row based on the picked card. If a card is picked, you last saw it at that pick number, and the other 7 people last saw it at picks p-1, p-2...,p-7. (or stopping earlier than p-7 if it's before pick 8) 
#This can be done on each row by adding (8 or p, whichever is smaller) to the num_counts of a card and sum_i_p-7_p(i) to its seen count
#then the average is seen_count/num_counts
#set up the summation as a couple dicts and a df to add them to later

#I don't think 17 lands includes when a card gets picked in the calculation, but it seems like an open question, 
#Based on the definition of 'last seen AT', i did include it, since the picker sees it)
#but you can change this value to 0 to not include it
AreWeIncludingWhenPickedInALSA=1

ALSA=ATA[['pick']]
#ALSA['SeenTotal']=0
#ALSA['SeenCount']=0
ALSA
#
SeenTotal={}
SeenCount={}
#add all the cards to the dicts
for r in ALSA.itertuples(index=False): 
    SeenTotal[r.pick]=0
    SeenCount[r.pick]=0
#loop over the picks, adding to the alsa summation based on the current pick
for r in Data.itertuples(index=False):   
    if(r.pick_number<8): #if <8 people saw it (no wheel)
        for i in range(1, r.pick_number+AreWeIncludingWhenPickedInALSA): #it was picks 1-PN 
            SeenTotal[r.pick] += i #add the total of pick numbers seen at to the total
            SeenCount[r.pick] += 1 #add the number of people who saw it to the count
    if(r.pick_number>=8): #if >8 people saw it (wheel)
        for i in range(r.pick_number-7, r.pick_number+AreWeIncludingWhenPickedInALSA): #it was last seen at PN and the seven before that
            SeenTotal[r.pick] += i #add the total of pick numbers seen at to the total
            SeenCount[r.pick] += 1 #add the number of people who saw it to the count

#add the values to the df (the dictionaries are mappings)
ALSA['SeenTotal']=ALSA['pick'].map(SeenTotal)
ALSA['SeenCount']=ALSA['pick'].map(SeenCount)
ALSA.head()


,pick,SeenTotal,SeenCount
0,Abrade,31356,10138
1,Abyssal Harvester,5540,2563
2,Adventuring Gear,106161,13255
3,Aegis Turtle,355487,42338
4,Aetherize,69375,11644


In [49]:
ALSA.head()

,pick,SeenTotal,SeenCount
0,Abrade,31356,10138
1,Abyssal Harvester,5540,2563
2,Adventuring Gear,106161,13255
3,Aegis Turtle,355487,42338
4,Aetherize,69375,11644


In [50]:
#now calculate the alsa as total/count (it's an average because both are summations)
ALSA['ALSA']=ALSA['SeenTotal']/ALSA['SeenCount']
ALSA.head()
ALSA.to_csv('../Outputs/FDN_ALSA.csv')